# RepairScope-X — Notebook 09
## First DLM-Native Selective-Remasking Smoke Test

This notebook marks the transition from the AR control experiments to a **real diffusion-language-model repair intervention**.

It is fully standalone and embeds three exact natural failures already used in the AR experiments.

### Why move forward even though Notebook 08's formal gate is false?

Notebook 08 completed a clean **200/200 independent trial matrix**. It reproduced:

- **3/7** preselected region interactions with `I >= 0.20`;
- replication across **2 datasets**;
- **2** interactions whose bootstrap 95% lower bound was above zero.

The formal gate failed only because global patch parsing was **95.00%** rather than the intentionally strict 98% threshold. Nine of the ten parse failures were concentrated in one GSM8K pair where the model repeatedly emitted an empty second patch; that pair did **not** drive the replicated positive interaction results.

So the AR mechanism evidence is developed enough for this stage. We now test the central diffusion-specific operation.

### Model

`Dream-org/Dream-v0-Instruct-7B`

The notebook uses **4-bit NF4 quantization only to make a first DLM runtime feasible on a Colab T4-class GPU**.

**Important:** these results are exploratory and must not be used as the final full-precision paper result. Quantization can change logits, confidence ordering, and denoising trajectories. A later final experiment should rerun the selected DLM conditions in full precision on a >=20 GB GPU.

### DLM-native repair mechanism

Dream initializes a generated response canvas with mask tokens and repeatedly denoises it.

For a failed response with token sequence `y`:

- `tail25`: force the first 75% of `y` at every denoising step and leave the last 25% masked;
- `tail50`: force the first 50% and leave the last 50% masked;
- `global`: leave the entire response canvas masked.

The forcing is applied through Dream's generation-token hook at initialization **and after every denoising step**. Thus untouched token positions cannot drift.

### Pilot matrix

3 natural failures × 3 repair scopes × 2 new seeds = **18 diffusion repair generations**

This notebook is a runtime/mechanism smoke test, not the final DLM benchmark.

In [ ]:
!pip -q install -U "transformers>=4.57,<5" "accelerate>=1.0" "bitsandbytes>=0.47" pandas

## 1. Runtime and GPU preflight

In [ ]:
import os, gc, json, math, re, time, random, hashlib, platform
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import torch
import transformers

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BASE_SEED = 2900
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)
    torch.cuda.empty_cache()

gc.collect()

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "Notebook 09 requires a CUDA GPU runtime."
)

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30

print("GPU:", GPU_NAME)
print("VRAM:", round(GPU_GB, 2), "GB")
print("Allocated:", round(torch.cuda.memory_allocated()/2**30, 3), "GB")

if GPU_GB < 14:
    raise RuntimeError(
        "This exploratory 4-bit Dream run expects approximately a T4-class "
        "GPU with ~15 GB VRAM or larger."
    )

if torch.cuda.memory_allocated()/2**30 > 1.0:
    raise RuntimeError(
        "GPU memory is already occupied. Restart the runtime and run from the top."
    )

print("Runtime preflight: PASS")

## 2. Recover the exact three natural failures

In [ ]:
import base64, zlib

EMBEDDED_PAYLOAD_SHA256 = "d441e967e92cf44adc2a2bcc0bca818c23ed82676e7457b03bbbf9531e3ca887"

_raw = zlib.decompress(
    base64.b64decode('''eNrtWGtv2zYU/SuEPrWD7ZKUZEnGHki7bAtQZJ2bbR/qgqUkyhYiiRpFxTWK/Pdd0nZix1LmdE0LFDViJ+J98PKec64Uf3AaUYhEi5RlPC9aJRpn8uaDUysZF6JkeepMnES2lU7lspokvMhjxXUuqwkJsT+hvjNwUq55I/SuJ8sg0wJsNVc6N/7GehttLEqWtQn6sxFILwSq2jIWqkEkGiAKb16liHpIvOeJLlZIVgn4SQRxaZusQzRXc6ER8UZoKnSrKsTRFeySIq5yvSiFzhNIUEMxDexqUwp4L8GI3v1ydn7ykp2cv/77dDpB36/3//HdCIpbJ3YmkBkPHNnqujWlPqERGkKBT9F3prRn6Al8DqHap7NqP9uQeJDmSqg8y5P1kSfQVt40Alqa8aIRA2iIFnOpVpB5qWQ1Z1ACRCQQ2Wj4rRlUnsJvGz5cF1MLVeZwnCvRZd0t/HrgQOtaXrBMqiVXKQMECgAYuuvkFRyJaXkpKlgh2B04c1EJgAe4sF02jkuIYY1IZJUax5Hn4dAdk4iQIHDHITUV8Ut2pXjJ5rEz8UfBOPQopiTyKHHDYOCUMhXFmkt/LEX1zHzQkT90nw/PKjhpm2hn66XEVd6s+cJ5KALqu0EUuTGPIjGO0ozzjPuRSDHxYxL42EsJxKq2Yk0+rziQQEBo6MUBocTLPE8I4cZRGuHITzIvyfw4SGmcBDxMMhoLklKfYozHmAZpRHDMxzjxLAKQDE7syEu4EkpJ5UyqtiiuB8fr4xHkQUEd3kYd4THq8B9NHf5ddYRGDJ5Vx11BfHI53CeGPSn490mBdEiBdkmBHEgBjyISRMQlUegFBI8jt1MKLg7JmI7DiOLoq5dCc8XLeqIVz0EMC14Mo7G7JwHrcBzxTzItFBKwXs2BuQsBIStUCl4Ydq8QdE4byjdiTffnbXIJrgteopoXPBEjdLEQSgzQlCcLUaACMlQiBWeukecj010tQVIJL614OrOAtKyhNnDItkEpX43QWYZCuBnUQtaFWGeymbsSGD3lmyS80Sj0TY4G/SaXqOTV6raQ+/PYQqB0CP5pR4VBuC/DC4myHGQMC2jRu8VO+lf76ZtaJEamZp8BdBlVwpQEjW5jQDbRO+MIyew2dQbIHbTKThU7jaQG5DrCJIwF67Hty02eo0NGs2pWvZYDuwZ0StrC0gnlzQRMszdoNtPivf7w102GNR67Z71GP2zdLuzGN9sBeJuNrmHA9aTaPfSO251U9mwS/NUm32z21hT/K4ywCmodot69oTygnHG5b2vwetiotfMThm3J1aVQDOBiR0xfo/7jxq+lZ//8peHBAKakcwAbz/0JPB5Fke+GxI/cAHsUbouHAzig8ONCzrHn4q/vWeTtwKmkFrGUlzhkc+iYQRvkL2q4lcOEZIAdtw99GEBpq/yfVqzXoAnbZehnlquSA6lWzDTXjmZrNVyBa5jggDHPFSwGdwJgljeCKbs33BR9aIGoC0M+gM/GsLyCWc4tPdhcMMwodibuzY2hYWZGsrthsL8ln2GjaUHewBMLFH5rhApU/h5OAifNtSllTXPAUdibzKaw9aZReCMArhncDhrN3I5d19HbdVMzT5RsGkbZtuCt093qGiOEBqrZOpxM2fT017Pfz9nZ+cXp9OTFhfl7evrq5dmLk4vTnzcVXe/haOpgNz22DPyU/5QpMbeHAuMU317H5towuM4LqXcxM8/9eLseC2ibGRmF2CbYlLpZZZw1bZLAU6XhAz4wxz1me+odk3dr2qvFWqBTGprPa5bkkc8KuTQGcmBY5POFsQSGDpBfVsWKpXmTSJVyA5PdpQSqGDIfmqE+xZewLqD81HDEVj2mQPOFLMr1tecGvpljHSxf0+C2qiKJ2VyzG4tNYuaFndcgWMywf8OK41H38KdHHY/G+6+ghwTk/5GA9pOA9JBg2AW2ZcGwix9bGtA+GpD7aUA7aRAGOyywEumkwGbmdHFgY/pUJPAfgwPundfjcODbIHgIB+ZNGV5u/tEaR94+7tb4H2jTIxU//uxjn/SiTfrQxr1oux+p9260v4TaH4w0ORbpL65r3Is0fjjSuA9p/HCkLbifHemDL08IIZ1fnjwm2vQb2l8I7b6vyj4K7SOn+ONpm/aiTR+Ott+HNv2YKU6PGuP9t+tj0H57/S/UEEwf''')
)

assert hashlib.sha256(_raw).hexdigest() == EMBEDDED_PAYLOAD_SHA256

UPSTREAM = json.loads(_raw.decode("utf-8"))
FAILURES = UPSTREAM["selected_failures"]
UPSTREAM_GATE = UPSTREAM["notebook08_gate"]

assert len(FAILURES) == 3
assert len({x["problem_id"] for x in FAILURES}) == 3
assert all(not x["verification"]["passed"] for x in FAILURES)

print("Payload SHA-256:", EMBEDDED_PAYLOAD_SHA256)

for f in FAILURES:
    print(
        f["problem_id"],
        "|",
        f["dataset"],
        "| target =",
        f["target"],
        "| original category =",
        f["verification"]["category"],
    )

print("Embedded failure payload: PASS")

## 3. Strict verifier

In [ ]:
FINAL_RE = re.compile(
    r"(?im)^[ \t]*FINAL_ANSWER[ \t]*:[ \t]*"
    r"([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)"
    r"[ \t]*$"
)

NUM_RE = re.compile(
    r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"
)

def strict_final(text):
    vals = FINAL_RE.findall(str(text))
    if not vals:
        return None

    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None


def permissive_last_number(text):
    vals = NUM_RE.findall(str(text).replace(",", ""))

    if not vals:
        return None

    try:
        x = float(vals[-1])
        return x if math.isfinite(x) else None
    except Exception:
        return None


def verify_text(text, target):
    strict = strict_final(text)
    loose = permissive_last_number(text)
    target = float(target)

    passed = (
        strict is not None
        and math.isclose(
            strict,
            target,
            abs_tol=1e-8,
            rel_tol=1e-8,
        )
    )

    return {
        "passed": bool(passed),
        "strict_prediction": strict,
        "permissive_prediction": loose,
        "target": target,
    }


assert verify_text("x\nFINAL_ANSWER: 4", 4)["passed"]
assert not verify_text("x\nFINAL_ANSWER: 5", 4)["passed"]

print("Verifier regression tests: PASS")

## 4. Experimental configuration

All three repair scopes use the **same 64 reverse denoising steps** so this smoke test has matched nominal DLM NFE.

Two independent stochastic seeds are used.

In [ ]:
MODEL_ID = "Dream-org/Dream-v0-Instruct-7B"

# Freeze the model revision used in the current public release.
MODEL_REVISION = "05334cb9faaf763692dcf9d8737c642be2b2a6ae"

ACTIONS = [
    "tail25",
    "tail50",
    "global",
]

SEEDS = [2901, 2902]

DENOISING_STEPS = 64
TEMPERATURE = 0.2
TOP_P = 0.95
ALG = "entropy"
ALG_TEMP = 0.0

EXPECTED_TRIALS = (
    len(FAILURES)
    * len(ACTIONS)
    * len(SEEDS)
)

print("Failures:", len(FAILURES))
print("Actions:", ACTIONS)
print("Seeds:", SEEDS)
print("Expected diffusion repair trials:", EXPECTED_TRIALS)

## 5. Load Dream 7B in exploratory 4-bit mode

This is deliberately an **exploratory hardware workaround**, not the paper's final precision condition.

In [ ]:
from transformers import (
    AutoModel,
    AutoTokenizer,
    BitsAndBytesConfig,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

torch.cuda.empty_cache()
gc.collect()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
)

model = AutoModel.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
).eval()

load_seconds = time.time() - t0

assert hasattr(model, "diffusion_generate"), (
    "Loaded model does not expose Dream's diffusion_generate()."
)

MODEL_DEVICE = next(model.parameters()).device

print("MODEL LOAD: PASS")
print("Class:", type(model).__name__)
print("Device:", MODEL_DEVICE)
print("Load seconds:", round(load_seconds, 2))
print("Allocated GB:", round(torch.cuda.memory_allocated()/2**30, 3))
print("Peak GB:", round(torch.cuda.max_memory_allocated()/2**30, 3))

## 6. Prepare prompt and failed-response token canvases

In [ ]:
SYSTEM_PROMPT = (
    "Solve the quantitative problem carefully. "
    "The final line must be exactly `FINAL_ANSWER: <number>` "
    "with no text after it."
)


def encode_problem_prompt(problem):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": problem,
        },
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    return {
        "input_ids": encoded.input_ids.to(MODEL_DEVICE),
        "attention_mask": encoded.attention_mask.to(MODEL_DEVICE),
    }


def encode_failed_response(text):
    ids = tokenizer(
        text,
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids[0]

    if len(ids) < 4:
        raise RuntimeError(
            "Failed response tokenization is unexpectedly short."
        )

    return ids


for f in FAILURES:
    ids = encode_failed_response(f["output"])
    print(
        f["problem_id"],
        "| response tokens =",
        len(ids),
    )

print("Token-canvas preparation: PASS")

## 7. Define DLM-native selective-remasking masks

`True` in `force_mask` means that token position is clamped to the original failed response at every denoising step.

The repair region is therefore the complement of `force_mask`.

In [ ]:
def force_mask_for_action(length, action):
    if action == "global":
        return torch.zeros(
            length,
            dtype=torch.bool,
        )

    if action == "tail50":
        cut = max(
            1,
            int(math.floor(length * 0.50)),
        )

        mask = torch.zeros(
            length,
            dtype=torch.bool,
        )
        mask[:cut] = True
        return mask

    if action == "tail25":
        cut = max(
            1,
            int(math.floor(length * 0.75)),
        )

        mask = torch.zeros(
            length,
            dtype=torch.bool,
        )
        mask[:cut] = True
        return mask

    raise ValueError(action)


for length in [8, 17, 40]:
    for action in ACTIONS:
        m = force_mask_for_action(
            length,
            action,
        )

        assert len(m) == length

        print(
            "length",
            length,
            "|",
            action,
            "| forced =",
            int(m.sum()),
            "| open =",
            int((~m).sum()),
        )

print("Selective-remasking mask audit: PASS")

## 8. Actual model-forward/NFE counter

In [ ]:
class ForwardCounter:
    def __init__(self, module):
        self.module = module
        self.calls = 0
        self.handle = None

    def _hook(self, module, inputs):
        self.calls += 1

    def __enter__(self):
        self.calls = 0
        self.handle = self.module.register_forward_pre_hook(
            self._hook
        )
        return self

    def __exit__(self, exc_type, exc, tb):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

## 9. DLM selective-remasking runner

The hook runs once before denoising and after each reverse step. It re-clamps every preserved token position to the original failed response.

A hard assertion verifies that the final sequence still contains the exact original token IDs at every forced position.

In [ ]:
def reseed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@torch.inference_mode()
def run_dlm_repair(
    failure,
    action,
    seed,
):
    reseed(seed)

    prompt = encode_problem_prompt(
        failure["prompt"]
    )

    original_ids = encode_failed_response(
        failure["output"]
    ).to(MODEL_DEVICE)

    response_len = int(
        original_ids.numel()
    )

    force_mask_cpu = force_mask_for_action(
        response_len,
        action,
    )

    force_mask = force_mask_cpu.to(
        MODEL_DEVICE
    )

    prompt_len = int(
        prompt["input_ids"].shape[1]
    )

    def clamp_hook(step, x, logits):
        generated = x[
            :,
            prompt_len:
            prompt_len + response_len
        ]

        if force_mask.any():
            generated[
                :,
                force_mask
            ] = original_ids[
                force_mask
            ].unsqueeze(0)

        return x

    torch.cuda.reset_peak_memory_stats()

    t0 = time.time()

    with ForwardCounter(model) as fc:
        output = model.diffusion_generate(
            prompt["input_ids"],
            attention_mask=
                prompt["attention_mask"],
            max_new_tokens=
                response_len,
            output_history=False,
            return_dict_in_generate=True,
            steps=DENOISING_STEPS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            alg=ALG,
            alg_temp=ALG_TEMP,
            generation_tokens_hook_func=
                clamp_hook,
        )

    wall = time.time() - t0

    final_ids = output.sequences[
        0,
        prompt_len:
        prompt_len + response_len
    ].detach()

    # Mechanical preservation audit.
    if force_mask.any():
        assert torch.equal(
            final_ids[force_mask],
            original_ids[force_mask],
        ), (
            "A forced token changed during DLM repair."
        )

    repaired = tokenizer.decode(
        final_ids.tolist(),
        skip_special_tokens=True,
    ).strip()

    verification = verify_text(
        repaired,
        failure["target"],
    )

    final_cpu = final_ids.cpu()
    original_cpu = original_ids.cpu()

    token_position_preservation = float(
        (
            final_cpu
            == original_cpu
        ).float().mean().item()
    )

    forced_fraction = float(
        force_mask_cpu.float().mean().item()
    )

    return {
        "problem_id":
            failure["problem_id"],
        "dataset":
            failure["dataset"],
        "action":
            action,
        "seed":
            int(seed),
        "response_tokens":
            response_len,
        "forced_tokens":
            int(
                force_mask_cpu.sum()
            ),
        "open_tokens":
            int(
                (~force_mask_cpu).sum()
            ),
        "forced_fraction":
            forced_fraction,
        "success":
            bool(
                verification["passed"]
            ),
        "verification":
            verification,
        "original_output":
            failure["output"],
        "repaired_output":
            repaired,
        "token_position_preservation":
            token_position_preservation,
        "actual_forward_calls":
            int(fc.calls),
        "wall_seconds":
            float(wall),
        "peak_vram_gb":
            float(
                torch.cuda.max_memory_allocated()
                / 2**30
            ),
    }

## 10. One real DLM selective-repair smoke test

In [ ]:
smoke = run_dlm_repair(
    FAILURES[0],
    action="tail25",
    seed=SEEDS[0],
)

print(
    json.dumps(
        {
            k: v
            for k, v in smoke.items()
            if k not in {
                "original_output",
                "repaired_output",
            }
        },
        indent=2,
    )
)

print("\nORIGINAL\n", smoke["original_output"])
print("\nDLM REPAIR\n", smoke["repaired_output"])

assert smoke["actual_forward_calls"] > 0
assert smoke["actual_forward_calls"] >= DENOISING_STEPS
assert math.isfinite(smoke["wall_seconds"])

print("\nREAL DLM SELECTIVE-REMASKING SMOKE: PASS")

## 11. Output directory and run signature

In [ ]:
SAVE_TO_DRIVE = True
CLEAR_OLD_RESULTS = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path(
        "/content/drive/MyDrive/RepairScopeX/Notebook09_Dream_DLM_Smoke"
    )
else:
    OUT_DIR = Path(
        "/content/RepairScopeX_Notebook09_Dream_DLM_Smoke"
    )

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

signature_payload = {
    "protocol_version":
        "dream-dlm-remask-smoke-v1",
    "embedded_payload_sha256":
        EMBEDDED_PAYLOAD_SHA256,
    "model_id":
        MODEL_ID,
    "model_revision":
        MODEL_REVISION,
    "quantization":
        "bnb_nf4_4bit_exploratory",
    "actions":
        ACTIONS,
    "seeds":
        SEEDS,
    "denoising_steps":
        DENOISING_STEPS,
    "temperature":
        TEMPERATURE,
    "top_p":
        TOP_P,
    "alg":
        ALG,
    "alg_temp":
        ALG_TEMP,
}

RUN_SIGNATURE = hashlib.sha256(
    json.dumps(
        signature_payload,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()

TRIALS_PATH = (
    OUT_DIR
    / "dream_dlm_repair_trials.jsonl"
)

MANIFEST_PATH = (
    OUT_DIR
    / "run_manifest.json"
)

if CLEAR_OLD_RESULTS:
    for p in [
        TRIALS_PATH,
        MANIFEST_PATH,
    ]:
        if p.exists():
            p.unlink()

if MANIFEST_PATH.exists():
    old = json.loads(
        MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )

    if old.get(
        "run_signature"
    ) != RUN_SIGNATURE:
        raise RuntimeError(
            "Existing Notebook 09 outputs use a different configuration."
        )

else:
    MANIFEST_PATH.write_text(
        json.dumps(
            {
                "created_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),
                "run_signature":
                    RUN_SIGNATURE,
                **signature_payload,
                "gpu":
                    GPU_NAME,
                "gpu_total_gb":
                    GPU_GB,
                "torch":
                    torch.__version__,
                "transformers":
                    transformers.__version__,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

print("Run signature:", RUN_SIGNATURE)
print("Output:", OUT_DIR)

## 12. Resume-safe 18-trial DLM repair sweep

In [ ]:
def trial_id(
    pid,
    action,
    seed,
):
    return (
        pid
        + "|"
        + action
        + "|"
        + str(seed)
    )


def load_completed(path):
    done = {}

    if not path.exists():
        return done

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        for line_number, line in enumerate(
            f,
            start=1,
        ):
            if not line.strip():
                continue

            rec = json.loads(line)

            if rec.get(
                "run_signature"
            ) != RUN_SIGNATURE:
                raise RuntimeError(
                    "Found trial from another Notebook 09 configuration."
                )

            tid = rec["trial_id"]

            if tid in done:
                raise RuntimeError(
                    f"Duplicate trial_id: {tid}"
                )

            done[tid] = rec

    return done


EXPECTED_IDS = {
    trial_id(
        f["problem_id"],
        action,
        seed,
    )
    for f in FAILURES
    for action in ACTIONS
    for seed in SEEDS
}

assert len(
    EXPECTED_IDS
) == EXPECTED_TRIALS

completed = load_completed(
    TRIALS_PATH
)

print(
    "Already completed:",
    len(completed),
    "/",
    EXPECTED_TRIALS,
)

counter = 0

with open(
    TRIALS_PATH,
    "a",
    encoding="utf-8",
) as fp:

    for failure in FAILURES:
        for action in ACTIONS:
            for seed in SEEDS:
                counter += 1

                tid = trial_id(
                    failure[
                        "problem_id"
                    ],
                    action,
                    seed,
                )

                if tid in completed:
                    print(
                        f"[{counter}/{EXPECTED_TRIALS}] SKIP {tid}"
                    )
                    continue

                try:
                    rec = run_dlm_repair(
                        failure,
                        action,
                        seed,
                    )

                    rec[
                        "trial_id"
                    ] = tid
                    rec[
                        "run_signature"
                    ] = RUN_SIGNATURE
                    rec[
                        "status"
                    ] = "ok"
                    rec[
                        "error"
                    ] = None

                except torch.cuda.OutOfMemoryError as e:
                    torch.cuda.empty_cache()
                    gc.collect()

                    rec = {
                        "trial_id":
                            tid,
                        "problem_id":
                            failure[
                                "problem_id"
                            ],
                        "dataset":
                            failure[
                                "dataset"
                            ],
                        "action":
                            action,
                        "seed":
                            seed,
                        "status":
                            "oom",
                        "error":
                            str(e),
                        "run_signature":
                            RUN_SIGNATURE,
                    }

                except Exception as e:
                    rec = {
                        "trial_id":
                            tid,
                        "problem_id":
                            failure[
                                "problem_id"
                            ],
                        "dataset":
                            failure[
                                "dataset"
                            ],
                        "action":
                            action,
                        "seed":
                            seed,
                        "status":
                            "error",
                        "error":
                            f"{type(e).__name__}: {e}",
                        "run_signature":
                            RUN_SIGNATURE,
                    }

                fp.write(
                    json.dumps(
                        rec,
                        ensure_ascii=False,
                    )
                )
                fp.write("\n")
                fp.flush()

                if rec["status"] == "ok":
                    print(
                        f"[{counter}/{EXPECTED_TRIALS}] "
                        f"{tid} | success={rec['success']} | "
                        f"NFE={rec['actual_forward_calls']} | "
                        f"pres={rec['token_position_preservation']:.3f}"
                    )
                else:
                    print(
                        f"[{counter}/{EXPECTED_TRIALS}] "
                        f"{tid} | {rec['status']} | {rec['error']}"
                    )

print("DLM sweep cell finished.")

## 13. Validate the DLM trial matrix

In [ ]:
records = load_completed(
    TRIALS_PATH
)

missing = (
    EXPECTED_IDS
    - set(records)
)

extra = (
    set(records)
    - EXPECTED_IDS
)

print("Expected:", len(EXPECTED_IDS))
print("Found:", len(records))
print("Missing:", len(missing))
print("Extra:", len(extra))

if missing:
    raise RuntimeError(
        "DLM sweep incomplete. Rerun the sweep cell; completed trials are skipped."
    )

if extra:
    raise RuntimeError(
        "Unexpected trial IDs exist."
    )

assert len(records) == 18

rows = [
    records[tid]
    for tid in sorted(
        EXPECTED_IDS
    )
]

status_counts = Counter(
    x["status"]
    for x in rows
)

print(
    "Statuses:",
    dict(status_counts),
)

if status_counts.get(
    "oom",
    0,
):
    raise RuntimeError(
        "At least one Dream DLM trial OOMed."
    )

if status_counts.get(
    "error",
    0,
):
    raise RuntimeError(
        "At least one Dream DLM trial errored."
    )

TRIAL_DF = pd.DataFrame(
    rows
)

assert len(TRIAL_DF) == 18
assert TRIAL_DF[
    "trial_id"
].nunique() == 18

print("18-trial DLM matrix integrity: PASS")

## 14. DLM repair-scope summary

In [ ]:
summary_rows = []

for action, group in TRIAL_DF.groupby(
    "action"
):
    summary_rows.append({
        "action":
            action,
        "trials":
            len(group),
        "repair_success_rate":
            float(
                group[
                    "success"
                ].mean()
            ),
        "mean_actual_nfe":
            float(
                group[
                    "actual_forward_calls"
                ].mean()
            ),
        "mean_preservation":
            float(
                group[
                    "token_position_preservation"
                ].mean()
            ),
        "mean_wall_seconds":
            float(
                group[
                    "wall_seconds"
                ].mean()
            ),
        "max_peak_vram_gb":
            float(
                group[
                    "peak_vram_gb"
                ].max()
            ),
    })

SUMMARY_DF = pd.DataFrame(
    summary_rows
).sort_values(
    "repair_success_rate",
    ascending=False,
)

display(SUMMARY_DF)

## 15. Per-failure DLM repair surface

With only two seeds, this is descriptive. The next notebook should expand only if the DLM runtime and preservation mechanism are sound.

In [ ]:
SURFACE_DF = (
    TRIAL_DF.groupby(
        [
            "problem_id",
            "dataset",
            "action",
        ],
        as_index=False,
    )
    .agg(
        success_rate=(
            "success",
            "mean",
        ),
        mean_nfe=(
            "actual_forward_calls",
            "mean",
        ),
        mean_preservation=(
            "token_position_preservation",
            "mean",
        ),
        mean_wall_seconds=(
            "wall_seconds",
            "mean",
        ),
    )
)

display(
    SURFACE_DF.sort_values(
        [
            "problem_id",
            "success_rate",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

## 16. DLM runtime/mechanism gate

The notebook distinguishes two things:

**Runtime ready**
- all 18 trials completed;
- no OOM/error;
- every trial executed at least 64 actual forward calls;
- every selective repair preserved all mechanically forced positions.

**Early repair signal**
- at least one of the 18 diffusion repairs passes the strict verifier.

A runtime-ready but zero-success result still tells us the DLM mechanism works; it means the next scientific issue is task/prompt/action design rather than infrastructure.

In [ ]:
forced_rows = TRIAL_DF[
    TRIAL_DF[
        "forced_tokens"
    ] > 0
].copy()

mechanical_preservation_ok = True

for _, row in forced_rows.iterrows():
    # Position preservation must be at least the forced fraction.
    if (
        row[
            "token_position_preservation"
        ]
        + 1e-12
        <
        row[
            "forced_fraction"
        ]
    ):
        mechanical_preservation_ok = False
        break


DLM_GATE = {
    "trials":
        int(
            len(TRIAL_DF)
        ),
    "unique_trial_ids":
        int(
            TRIAL_DF[
                "trial_id"
            ].nunique()
        ),
    "successful_repairs":
        int(
            TRIAL_DF[
                "success"
            ].sum()
        ),
    "at_least_one_success":
        bool(
            TRIAL_DF[
                "success"
            ].any()
        ),
    "minimum_actual_nfe":
        int(
            TRIAL_DF[
                "actual_forward_calls"
            ].min()
        ),
    "mechanical_preservation_ok":
        bool(
            mechanical_preservation_ok
        ),
    "no_runtime_errors":
        True,
    "quantization":
        "bnb_nf4_4bit_exploratory",
}

DLM_GATE[
    "DLM_RUNTIME_READY"
] = bool(
    DLM_GATE[
        "trials"
    ] == 18
    and DLM_GATE[
        "unique_trial_ids"
    ] == 18
    and DLM_GATE[
        "minimum_actual_nfe"
    ] >= DENOISING_STEPS
    and DLM_GATE[
        "mechanical_preservation_ok"
    ]
)

DLM_GATE[
    "DLM_EARLY_REPAIR_SIGNAL"
] = bool(
    DLM_GATE[
        "DLM_RUNTIME_READY"
    ]
    and DLM_GATE[
        "at_least_one_success"
    ]
)

print(
    json.dumps(
        DLM_GATE,
        indent=2,
    )
)

## 17. Save artifacts

In [ ]:
summary_path = (
    OUT_DIR
    / "dlm_action_summary.csv"
)

surface_path = (
    OUT_DIR
    / "dlm_repair_surface.csv"
)

gate_path = (
    OUT_DIR
    / "DLM_SMOKE_GATE.json"
)

SUMMARY_DF.to_csv(
    summary_path,
    index=False,
)

SURFACE_DF.to_csv(
    surface_path,
    index=False,
)

gate_path.write_text(
    json.dumps(
        DLM_GATE,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Saved Notebook 09 artifacts:",
    OUT_DIR,
)

## 18. Artifact hashes

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as f:
        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


artifact_paths = [
    MANIFEST_PATH,
    TRIALS_PATH,
    summary_path,
    surface_path,
    gate_path,
]

artifact_manifest = {
    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
    "run_signature":
        RUN_SIGNATURE,
    "embedded_payload_sha256":
        EMBEDDED_PAYLOAD_SHA256,
    "files": {
        p.name:
            sha256_file(p)
        for p in artifact_paths
        if p.exists()
    },
}

artifact_manifest_path = (
    OUT_DIR
    / "artifact_manifest.json"
)

artifact_manifest_path.write_text(
    json.dumps(
        artifact_manifest,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        artifact_manifest,
        indent=2,
    )
)

## 19. ZIP and download all Notebook 09 results

In [ ]:
import shutil

DOWNLOAD_RESULTS = True

ZIP_BASE = Path(
    "/content/RepairScopeX_Notebook09_Dream_DLM_Smoke_results"
)

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=OUT_DIR,
)

print("Created:", zip_path)
print(
    "Size MB:",
    round(
        Path(
            zip_path
        ).stat().st_size
        / 1024**2,
        3,
    ),
)

if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(
        zip_path
    )

# Interpretation / next progression

If `DLM_RUNTIME_READY = true`, the central selective-remasking mechanism is functioning on a real diffusion LM.

If `DLM_EARLY_REPAIR_SIGNAL = true`, Notebook 10 should scale this to:

- a larger natural-failure set;
- 5+ repair seeds;
- finer repair scopes;
- matched actual NFE;
- fresh/global regeneration control;
- repair energy and same-error recurrence.

If the runtime is ready but the early repair signal is zero, do **not** immediately discard the idea. First inspect the three repaired trajectories and determine whether the fixed-length canvas, prompt framing, or repair-region definition is the limiting factor.

The 4-bit results remain exploratory. Final paper evidence must include at least one full-precision DLM condition.